# Theoretical Extensions — DCF

Analytical companion to empirical notebooks. No GPU required.

1. Lemma 1: Lipschitz Consistency Bound (S >= 1 - L*eps/margin)
2. Lemma 2: CAG Dichotomy — all 16 configurations are Overconfident or Consistently Wrong
3. Architecture scaling: CS vs depth and parameters
4. Medical case study: OCTMNIST CAG reveals dangerous failure modes
5. Master results table + LaTeX export
6. Perturbation flip rate analysis

In [ ]:
OUTPUT_DIR='E:\\decision_consistency_analysis'
import os, csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
os.makedirs(os.path.join(OUTPUT_DIR,'figures'),exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR,'tables'),exist_ok=True)
print('Output:',OUTPUT_DIR)

In [ ]:
# Lemma 1: Lipschitz Consistency Bound
# For classifier with Lipschitz constant L and decision margin m:
# Label Stability S >= 1 - L*epsilon/m
def lipschitz_min_cs(L,eps,margin): return 1-min(1.0,(L*eps)/margin)
eps,margin=0.01,0.1
print('Lemma 1 - Lipschitz Consistency Bound (eps=0.01, margin=0.1):')
print(f'{"L":>6}  {"Min CS":>8}  {"Max Flip":>10}')
for L in [1.0,2.0,5.0,10.0]:
    min_cs=lipschitz_min_cs(L,eps,margin)
    print(f'{L:>6.1f}  {min_cs:>8.4f}  {1-min_cs:>10.4f}')
L_range=np.linspace(0.5,20,200); cs_bounds=[lipschitz_min_cs(L,eps,margin) for L in L_range]
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(L_range,cs_bounds,linewidth=2,color='steelblue'); ax.fill_between(L_range,cs_bounds,1,alpha=0.15,color='steelblue')
ax.axhline(0.8,color='tomato',linestyle='--',lw=1.5,label='Typical threshold 0.8')
ax.set_xlabel('Lipschitz Constant L',fontsize=11); ax.set_ylabel('Minimum Guaranteed CS',fontsize=11)
ax.set_title('Lemma 1: CS Lower Bound vs Lipschitz Constant',fontsize=11); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'figures','lemma1_lipschitz_bound.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
# All 16 CNN configurations — empirical results
all_results=[
    ('CIFAR-10','ResNet-18',0.876,0.7822,0.0938),('CIFAR-10','ResNet-50',0.865,0.7889,0.0761),
    ('CIFAR-10','VGG-16',0.856,0.7767,0.0793),('CIFAR-10','MobileNetV2',0.874,0.7757,0.0983),
    ('STL-10','ResNet-18',0.939,0.8571,0.0819),('STL-10','ResNet-50',0.969,0.8730,0.0960),
    ('STL-10','VGG-16',0.936,0.8383,0.0977),('STL-10','MobileNetV2',0.935,0.8538,0.0812),
    ('Intel-Scene','ResNet-18',0.574,0.8659,-0.2919),('Intel-Scene','ResNet-50',0.526,0.8478,-0.3218),
    ('Intel-Scene','VGG-16',0.742,0.8484,-0.1064),('Intel-Scene','MobileNetV2',0.599,0.8388,-0.2398),
    ('OCTMNIST','ResNet-18',0.470,0.8612,-0.3912),('OCTMNIST','ResNet-50',0.526,0.8466,-0.3206),
    ('OCTMNIST','VGG-16',0.628,0.8303,-0.2023),('OCTMNIST','MobileNetV2',0.513,0.8146,-0.3016),
]
print(f'{"Dataset":<14} {"Model":<14} {"Acc":>6} {"CS":>7} {"CAG":>8}  Mode')
print('-'*68)
for ds,model,acc,cs,cag in all_results:
    print(f'{ds:<14} {model:<14} {acc:>6.3f} {cs:>7.4f} {cag:>8.4f}  {"Overconfident" if cag>0 else "Consistently Wrong"}')
overconf=sum(1 for *_,cag in all_results if cag>0); cw=sum(1 for *_,cag in all_results if cag<0)
print(f'\nLemma 2 CAG Dichotomy: Overconfident={overconf}, Consistently Wrong={cw}, Well-Calibrated=0')

In [ ]:
# Lemma 2 visualisation
configs=[f'{ds[:4]}-{model[:8]}' for ds,model,*_ in all_results]
cag_vals=[cag for *_,cag in all_results]
fig,ax=plt.subplots(figsize=(13,6))
ax.barh(configs,cag_vals,color=['tomato' if c>0 else 'steelblue' for c in cag_vals])
ax.axvline(0,color='black',linewidth=2); ax.set_xlabel('CAG (Accuracy minus CS)',fontsize=11)
ax.set_title('Lemma 2: CAG Dichotomy — All 16 CNN Configurations\nRed=Overconfident  Blue=Consistently Wrong',fontsize=11); ax.grid(axis='x',alpha=0.3); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'figures','lemma2_cag_dichotomy.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
arch={'ResNet-18':{'depth':18,'params_m':11.7},'ResNet-50':{'depth':50,'params_m':25.6},'VGG-16':{'depth':16,'params_m':138.4},'MobileNetV2':{'depth':53,'params_m':3.5}}
models_unique=['ResNet-18','ResNet-50','VGG-16','MobileNetV2']; datasets_unique=['CIFAR-10','STL-10']
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ds in datasets_unique:
    ds_rows=[(model,cs) for d,model,acc,cs,cag in all_results if d==ds]
    axes[0].scatter([arch[m]['depth'] for m,_ in ds_rows],[cs for _,cs in ds_rows],s=120,label=ds)
    axes[1].scatter([arch[m]['params_m'] for m,_ in ds_rows],[cs for _,cs in ds_rows],s=120,label=ds)
axes[0].set_xlabel('Depth'); axes[0].set_ylabel('CS'); axes[0].set_title('CS vs Depth'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('Params (M)'); axes[1].set_xscale('log'); axes[1].set_title('Efficiency: CS vs Params'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); path=os.path.join(OUTPUT_DIR,'figures','architecture_scaling.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
# Medical case study: OCTMNIST ResNet-18
# Accuracy=47% looks like 'needs improvement'; CAG=-0.39 reveals systematic consistent wrong predictions
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].bar(['Accuracy','Consistency Score'],[0.470,0.861],color=['tomato','steelblue'])
axes[0].set_ylim(0,1); axes[0].set_title('OCTMNIST/ResNet-18: Standard vs DCF'); [axes[0].text(x,v+0.02,f'{v:.3f}',ha='center') for x,v in enumerate([0.470,0.861])]
axes[1].bar(['CAG'],[-0.3912],color='steelblue'); axes[1].axhline(0,color='black',lw=2)
axes[1].set_ylim(-0.6,0.2); axes[1].set_title('CAG=-0.391: Consistently Wrong'); axes[1].text(0,-0.45,'CONSISTENTLY WRONG',ha='center',color='tomato',fontweight='bold')
fig.suptitle('How DCF Reveals Clinical Deployment Risk'); plt.tight_layout(); path=os.path.join(OUTPUT_DIR,'figures','medical_case_study.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
# LaTeX master results table
latex=[r'\begin{table}[htbp]',r'\centering',r'\caption{Complete CNN Results - DCF}',r'\label{tab:master}',r'\begin{tabular}{llcccl}',r'\toprule',r'\textbf{Dataset} & \textbf{Model} & \textbf{Acc} & \textbf{CS} & \textbf{CAG} & \textbf{Mode} \\',r'\midrule']
for ds,model,acc,cs,cag in sorted(all_results,key=lambda x:x[4],reverse=True):
    mode='Overconfident' if cag>0 else 'Consistently Wrong'
    latex.append(f'{ds} & {model} & {acc:.3f} & {cs:.4f} & {cag:.4f} & {mode} \\\\')
latex+=[r'\bottomrule',r'\end{tabular}',r'\end{table}']
tex_path=os.path.join(OUTPUT_DIR,'tables','master_results.tex')
with open(tex_path,'w') as f: f.write('\n'.join(latex))
print('LaTeX saved:',tex_path)